In [8]:
from pyiceberg.catalog.rest import RestCatalog

catalog = RestCatalog(
    name = "catalog",
    warehouse = "bde-warehouse",
    uri = "http://localhost:8181/catalog",
    token = "dummy",
)

In [ ]:
# Create namespace
DAFT_NAMESPACE = "daft_namespace"

if (DAFT_NAMESPACE,) not in catalog.list_namespaces():
    catalog.create_namespace(DAFT_NAMESPACE)

In [ ]:
# Create table
from pyiceberg.schema import Schema
from pyiceberg.types import (
    TimestampType,
    FloatType,
    DoubleType,
    StringType,
    NestedField,
)

from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.table.sorting import SortOrder, SortField

TABLE_NAME = "bids"

schema = Schema(
    NestedField(field_id=1, name="datetime", field_type=TimestampType(), required=True),
    NestedField(field_id=2, name="symbol", field_type=StringType(), required=True),
    NestedField(field_id=3, name="bid", field_type=FloatType(), required=False),
    NestedField(field_id=4, name="ask", field_type=DoubleType(), required=False),
)

partition_spec = PartitionSpec(
    PartitionField(
        source_id=1, field_id=1000, transform="day", name="datetime_day"
    )
)

# Sort on the symbol
sort_order = SortOrder(SortField(source_id=2, transform='identity'))

catalog.create_table(
    identifier=f"{DAFT_NAMESPACE}.{TABLE_NAME}",
    schema=schema,
    partition_spec=partition_spec,
    sort_order=sort_order,
)

In [ ]:
# Reading data trough pyiceberg and Daft
import daft

table = catalog.load_table(f"{DAFT_NAMESPACE}.{TABLE_NAME}")
df = daft.read_iceberg(table)
df.show()

datetimeTimestamp[us],symbolString,bidFloat32,askFloat64
